[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Valdefer54/Hunt-and-Kill-y-Busqueda-No-Informada/blob/main/Taller_Hunt_and_Kill_Busqueda_No_Informada.ipynb)
[![View on nbviewer](https://img.shields.io/badge/render-nbviewer-orange.svg)](https://nbviewer.org/github/Valdefer54/Hunt-and-Kill-y-Busqueda-No-Informada/blob/main/Taller_Hunt_and_Kill_Busqueda_No_Informada.ipynb)

# Hunt-and-Kill y Búsqueda no Informada

**Curso:** Inteligencia Artificial · **Referencia:** AIMA, capítulo 3  
**Modalidad:** individual · **Duración sugerida:** 3 semanas  
**Producto:** un cuaderno Jupyter con tres implementaciones comparables

## Problema

Se proporciona solamente un generador de laberintos perfectos mediante **Hunt-and-Kill**. A partir del grafo producido, el estudiante deberá diseñar, implementar, verificar y comparar algoritmos de búsqueda no informada para encontrar un camino entre dos celdas.

El proyecto debe realizarse en tres versiones:

1. **Desde cero:** sin bibliotecas de búsqueda ni de grafos.
2. **SimpleAI:** modelando el laberinto como un `SearchProblem`.
3. **AIMA-Python:** modelándolo como una subclase de `Problem`.

> Este cuaderno es deliberadamente incompleto. No contiene implementaciones Python de DFS, BFS, UCS, IDDFS, búsqueda bidireccional ni Lee. El trabajo evaluado consiste en convertir las especificaciones y el pseudocódigo en soluciones propias.

In [ ]:
# Configuración automática para Google Colab
# Si ejecutas en Colab, esta celda instala simpleai y descarga los módulos del repositorio
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Google Colab detectado. Instalando dependencias y módulos...")
    import os, subprocess
    subprocess.run(["pip", "install", "-q", "simpleai"], check=True)
    if not os.path.exists("src"):
        os.system("git clone https://github.com/Valdefer54/Hunt-and-Kill-y-Busqueda-No-Informada.git _repo")
        os.system("cp -r _repo/src _repo/aima _repo/datos_experimentos.csv _repo/*.png . 2> /dev/null || true")
        os.system("rm -rf _repo")
    print("✓ Entorno Colab listo con todas las dependencias y módulos.")


## Resultados de aprendizaje

Al finalizar, el estudiante estará en capacidad de:

- formular un entorno como espacio de estados;
- separar el problema de la estrategia de búsqueda;
- implementar búsqueda en árbol y búsqueda en grafo;
- justificar completitud, optimalidad y complejidad;
- adaptar un mismo problema a dos bibliotecas de IA;
- construir experimentos reproducibles;
- interpretar el algoritmo de Lee como BFS por frentes de onda;
- defender decisiones de diseño y diagnosticar errores sin depender de una solución externa.

## Condiciones y restricciones

- No se permite `networkx` ni funciones externas que ya resuelvan caminos.
- En la versión desde cero no se permite SimpleAI, AIMA-Python ni otra biblioteca de búsqueda.
- Los estados deben ser inmutables y utilizables como claves de diccionario.
- Cada algoritmo debe devolver solución y métricas, no solamente imprimirlas.
- Todas las versiones deben trabajar con el **mismo grafo**, inicio, meta y costos.
- Los resultados deben ser reproducibles mediante semillas.
- No se acepta una imagen como evidencia: deben entregarse estructuras de datos verificables.
- El estudiante debe citar cualquier recurso consultado y conservar una bitácora breve de decisiones.

### Evidencia de autoría y comprensión

La evaluación incluye:

1. historial de al menos seis hitos o commits;
2. una traza manual sobre un laberinto pequeño;
3. defensa oral individual de 5–8 minutos;
4. modificación en vivo de inicio, meta, semilla o costo;
5. explicación de una decisión que produjo un error y cómo se corrigió.

Durante la defensa se podrá solicitar reconstruir una parte pequeña del algoritmo sin consultar el cuaderno.

## 1. Código suministrado: generador Hunt-and-Kill

Este es el único algoritmo completo entregado. El resultado es un diccionario de adyacencia no dirigido:

```text
grafo[celda] = conjunto de celdas conectadas por un corredor
```

Una celda se representa como `(fila, columna)`.

In [ ]:
import random


class LaberintoHuntKill:
    """Genera un laberinto perfecto como grafo no dirigido."""

    DIRECCIONES = ((-1, 0), (1, 0), (0, -1), (0, 1))

    def __init__(self, filas=25, columnas=25, semilla=2026):
        self.filas = filas
        self.columnas = columnas
        self.rng = random.Random(semilla)
        self.grafo = {
            (fila, columna): set()
            for fila in range(filas)
            for columna in range(columnas)
        }

    def vecinos_geometricos(self, celda):
        fila, columna = celda
        for df, dc in self.DIRECCIONES:
            vecino = (fila + df, columna + dc)
            if (0 <= vecino[0] < self.filas and
                    0 <= vecino[1] < self.columnas):
                yield vecino

    def conectar(self, origen, destino):
        self.grafo[origen].add(destino)
        self.grafo[destino].add(origen)

    def generar(self):
        no_visitadas = set(self.grafo)
        actual = self.rng.choice(tuple(no_visitadas))
        no_visitadas.remove(actual)

        while no_visitadas:
            # KILL: avanzar aleatoriamente hacia una celda no visitada.
            libres = [
                vecino
                for vecino in self.vecinos_geometricos(actual)
                if vecino in no_visitadas
            ]

            if libres:
                siguiente = self.rng.choice(libres)
                self.conectar(actual, siguiente)
                no_visitadas.remove(siguiente)
                actual = siguiente
                continue

            # HUNT: localizar una celda libre vecina del árbol construido.
            candidatas = []
            for celda in no_visitadas:
                visitados = [
                    vecino
                    for vecino in self.vecinos_geometricos(celda)
                    if vecino not in no_visitadas
                ]
                if visitados:
                    candidatas.append((celda, visitados))

            actual, visitados = self.rng.choice(candidatas)
            vecino = self.rng.choice(visitados)
            self.conectar(actual, vecino)
            no_visitadas.remove(actual)

        return self.grafo

### Actividad 1 — Auditoría del generador

Antes de resolver el laberinto:

1. Explique las fases Hunt y Kill señalando las instrucciones correspondientes.
2. Demuestre que el grafo generado es conexo.
3. Justifique por qué contiene exactamente $|V|-1$ aristas.
4. Explique por qué esas dos propiedades implican que es un árbol.
5. Diseñe pruebas para comprobar:
   - simetría de las adyacencias;
   - ausencia de conexiones diagonales;
   - validez de coordenadas;
   - conectividad;
   - ausencia de ciclos;
   - reproducibilidad con una semilla.

No basta con ejecutar el generador: las propiedades deben verificarse automáticamente.

**Respuesta y pruebas del estudiante:**

### Actividad 1 — Auditoría del generador Hunt-and-Kill

#### 1. ¿Cómo funcionan las fases Hunt y Kill en el código suministrado?
Revisando directamente el código de la clase `LaberintoHuntKill`:

- Arranque inicial (líneas 139-142): Al comenzar, metemos todas las celdas de la grilla en el conjunto `no_visitadas`. Con `self.rng.choice(tuple(no_visitadas))` escogemos una celda al azar, la sacamos de `no_visitadas` y esa será la primera celda de nuestro laberinto.
- Fase KILL (Caminar abriendo camino, líneas 144-156):
  ```python
  libres = [vecino for vecino in self.vecinos_geometricos(actual) if vecino in no_visitadas]
  if libres:
      siguiente = self.rng.choice(libres)
      self.conectar(actual, siguiente)
      no_visitadas.remove(siguiente)
      actual = siguiente
      continue
  ```
  Estando en la celda `actual`, miramos los vecinos ortogonales (arriba, abajo, izquierda, derecha) que sigan en `no_visitadas`. Si hay al menos uno libre, elegimos uno al azar (`siguiente`), abrimos el pasillo con `self.conectar()`, lo sacamos de `no_visitadas` y nos movemos a esa celda. Hacemos esto una y otra vez mientras tengamos vecinos libres hacia donde seguir abriendo camino.
- Fase HUNT (Buscar enganche cuando nos quedamos atrapados, líneas 158-173):
  ```python
  candidatas = []
  for celda in no_visitadas:
      visitados = [vecino for vecino in self.vecinos_geometricos(celda) if vecino not in no_visitadas]
      if visitados:
          candidatas.append((celda, visitados))
  actual, visitados = self.rng.choice(candidatas)
  vecino = self.rng.choice(visitados)
  self.conectar(actual, vecino)
  no_visitadas.remove(actual)
  ```
  Cuando caemos en un callejón sin salida (ya no hay vecinos libres alrededor de `actual`), entramos a la fase Hunt. Aquí revisamos las celdas que todavía no hemos visitado y buscamos cuáles de ellas están pegadas a alguna celda que YA pertenezca al laberinto. De esas candidatas, elegimos una al azar, la conectamos con un pasillo a su vecino ya visitado para engancharla al laberinto, y a partir de esa celda arranca otra vez una fase Kill en la siguiente vuelta del `while`.

El algoritmo termina cuando `no_visitadas` queda totalmente vacío.

#### 2. Demostración de que el grafo siempre es conexo
La demostración sale de forma natural por inducción:
- Caso base: Al inicio tenemos una sola celda en el laberinto, la cual por sí sola forma un grafo conexo.
- Paso inductivo: Supongamos que el laberinto que llevamos construido hasta el momento es conexo. Cada vez que añadimos una celda nueva (ya sea en fase Kill o en fase Hunt), el algoritmo crea una arista conectándola directamente a una celda que ya formaba parte del laberinto.
- Como la celda vieja ya tenía camino hacia todas las demás celdas del laberinto, la celda nueva también queda conectada con todo el resto. Por tanto, el grafo sigue siendo conexo en cada paso.
- Como el bucle no se detiene hasta visitar todas las celdas de la grilla ($|V|$), al final todas quedan unidas en un único bloque conexo. Es imposible que quede ninguna celda aislada o desconectada.

#### 3. Justificación de por qué tiene exactamente $|V| - 1$ aristas
- Empezamos con 1 celda y 0 aristas.
- Cada vez que se llama a `self.conectar()`, se añade exactamente una arista no dirigida y se saca exactamente una celda de `no_visitadas`.
- Para visitar las $|V|$ celdas partiendo de la primera, tenemos que incorporar exactamente $|V| - 1$ celdas nuevas.
- Como cada celda nueva aporta exactamente una arista al laberinto, al final tendremos obligatoriamente $|E| = |V| - 1$ aristas.

#### 4. ¿Por qué estas dos cosas garantizan que es un árbol?
En teoría de grafos hay una propiedad clásica: si un grafo con $|V|$ nodos es conexo y tiene exactamente $|V| - 1$ aristas, entonces obligatoriamente es un árbol. Esto implica que no tiene ningún ciclo ni bucle cerrado, y que entre cualquier par de celdas existe un único camino simple. Por esta razón el resultado es un laberinto perfecto.


In [ ]:
# Verificación y Auditoría Automática del Generador Hunt-and-Kill
from collections import deque
import random

def auditar_laberinto(laberinto_cls, filas=20, columnas=20, semilla=2026):
    lab = laberinto_cls(filas=filas, columnas=columnas, semilla=semilla)
    grafo = lab.generar()
    total_nodos = filas * columnas
    
    print(f"=== AUDITORÍA AUTOMÁTICA DEL GENERADOR ({filas}x{columnas}, semilla={semilla}) ===\n")
    
    # 1. Simetría de adyacencias
    simetrico = True
    for u, vecinos in grafo.items():
        for v in vecinos:
            if u not in grafo[v]:
                simetrico = False
                break
    print(f"[1] Simetría de adyacencias: {'✓ CORRECTO' if simetrico else '✗ FALLÓ'}")
    assert simetrico, "Error: Hay aristas no dirigidas asimétricas."
    
    # 2. Ausencia de diagonales (ortogonalidad)
    ortogonal = True
    for u, vecinos in grafo.items():
        for v in vecinos:
            manhattan = abs(u[0] - v[0]) + abs(u[1] - v[1])
            if manhattan != 1:
                ortogonal = False
                break
    print(f"[2] Ausencia de diagonales (distancia Manhattan = 1): {'✓ CORRECTO' if ortogonal else '✗ FALLÓ'}")
    assert ortogonal, "Error: Se detectaron conexiones diagonales o no unitarias."
    
    # 3. Validez de coordenadas
    coordenadas_validas = all(
        0 <= r < filas and 0 <= c < columnas
        for r, c in grafo
        for vr, vc in grafo[(r, c)]
        if 0 <= vr < filas and 0 <= vc < columnas
    )
    print(f"[3] Validez de coordenadas dentro de [0, {filas})x[0, {columnas}): {'✓ CORRECTO' if coordenadas_validas else '✗ FALLÓ'}")
    assert coordenadas_validas, "Error: Coordenadas fuera de rango."
    
    # 4. Conectividad total (BFS)
    inicio = next(iter(grafo))
    visitados = set([inicio])
    cola = deque([inicio])
    while cola:
        curr = cola.popleft()
        for vec in grafo[curr]:
            if vec not in visitados:
                visitados.add(vec)
                cola.append(vec)
    conexo = (len(visitados) == total_nodos)
    print(f"[4] Conectividad global (|V_alcanzados| = {len(visitados)}/{total_nodos}): {'✓ CORRECTO' if conexo else '✗ FALLÓ'}")
    assert conexo, f"Error: Grafo desconexo ({len(visitados)}/{total_nodos} nodos alcanzados)."
    
    # 5. Ausencia de ciclos y fórmula de árbol (|E| == |V| - 1)
    num_aristas = sum(len(vecinos) for vecinos in grafo.values()) // 2
    es_arbol_aristas = (num_aristas == total_nodos - 1)
    
    # Detección explícita de ciclos vía DFS
    hay_ciclo = False
    padres = {}
    visitados_dfs = set()
    pila = [(inicio, None)]
    while pila:
        curr, p = pila.pop()
        if curr in visitados_dfs:
            continue
        visitados_dfs.add(curr)
        padres[curr] = p
        for vec in grafo[curr]:
            if vec == p:
                continue
            if vec in visitados_dfs:
                hay_ciclo = True
                break
            pila.append((vec, curr))
        if hay_ciclo:
            break
            
    print(f"[5] Ausencia de ciclos (|E| = {num_aristas}, |V|-1 = {total_nodos-1}, ciclos={hay_ciclo}): {'✓ CORRECTO' if (es_arbol_aristas and not hay_ciclo) else '✗ FALLÓ'}")
    assert es_arbol_aristas and not hay_ciclo, "Error: El grafo contiene ciclos o viola |E| = |V| - 1."
    
    # 6. Reproducibilidad con semilla
    lab_rep = laberinto_cls(filas=filas, columnas=columnas, semilla=semilla)
    grafo_rep = lab_rep.generar()
    reproducible = (grafo == grafo_rep)
    
    lab_dif = laberinto_cls(filas=filas, columnas=columnas, semilla=semilla + 99)
    grafo_dif = lab_dif.generar()
    sensible_semilla = (grafo != grafo_dif)
    
    print(f"[6] Reproducibilidad por semilla: {'✓ CORRECTO' if (reproducible and sensible_semilla) else '✗ FALLÓ'}")
    assert reproducible and sensible_semilla, "Error en reproducibilidad de semillas."
    
    print("\n>>> TODAS LAS PROPIEDADES FUERON VERIFICADAS Y AUDITADAS EXITOSAMENTE <<<")

# Ejecutar auditoría sobre el generador suministrado
auditar_laberinto(LaberintoHuntKill, filas=20, columnas=20, semilla=2026)


## 2. Instancia individual

Cada estudiante debe construir una instancia reproducible:

- `semilla = últimos 5 dígitos del código estudiantil`;
- `filas = 20 + último dígito`;
- `columnas = 20 + penúltimo dígito`;
- inicio y meta deben estar en cuadrantes opuestos, pero no necesariamente en las esquinas.

### Parámetros asignados

| Parámetro | Valor |
|---|---|
| Código estudiantil | `1077431204` |
| Filas ($20 + 7$) | `27` |
| Columnas ($20 + 5$) | `25` |
| Inicio (Cuadrante Superior Izquierdo) | `(3, 4)` |
| Meta (Cuadrante Inferior Derecho) | `(23, 20)` |

> *Nota:* El código está parametrizado mediante variables explícitas, lo que permite al docente modificar cualquiera de estos valores en vivo durante la sustentación sin alterar la lógica de búsqueda.

In [ ]:
# Construcción reproducible de la instancia individual
CODIGO_ESTUDIANTIL = 1077431204
SEMILLA = int(str(CODIGO_ESTUDIANTIL)[-5:])       
FILAS = 20 + int(str(CODIGO_ESTUDIANTIL)[-1])      
COLUMNAS = 20 + int(str(CODIGO_ESTUDIANTIL)[-2])   

# Cuadrantes opuestos: Superior Izquierdo e Inferior Derecho
INICIO = (3, 4)
META = (FILAS - 4, COLUMNAS - 5)  

# Instanciar y generar el laberinto
laberinto_individual = LaberintoHuntKill(filas=FILAS, columnas=COLUMNAS, semilla=SEMILLA)
grafo_individual = laberinto_individual.generar()

# Verificación de cuadrantes
def obtener_cuadrante(celda, filas, columnas):
    r, c = celda
    vertical = "Norte (Superior)" if r < filas / 2 else "Sur (Inferior)"
    horizontal = "Oeste (Izquierdo)" if c < columnas / 2 else "Este (Derecho)"
    return f"{vertical} - {horizontal}"

print("=== INSTANCIA INDIVIDUAL CONFIGURADA ===")
print(f"Código: {CODIGO_ESTUDIANTIL}")
print(f"Semilla: {SEMILLA}")
print(f"Dimensiones: {FILAS} filas x {COLUMNAS} columnas ({FILAS * COLUMNAS} celdas)")
print(f"Inicio: {INICIO} -> Cuadrante: {obtener_cuadrante(INICIO, FILAS, COLUMNAS)}")
print(f"Meta:   {META} -> Cuadrante: {obtener_cuadrante(META, FILAS, COLUMNAS)}")
print(f"Total aristas en el árbol: {sum(len(v) for v in grafo_individual.values()) // 2} (|V|-1)")


## 3. Formulación formal del problema

Modelamos el laberinto como un problema de búsqueda tradicional (AIMA, Cap. 3):

| Componente | Definición en nuestro laberinto |
|---|---|
| Estados ($S$) | Cada celda es una tupla inmutable de coordenadas: `(fila, columna)`. |
| Estado inicial ($s_0$) | La celda donde arranca el agente: `(3, 4)`. |
| Acciones | Moverse a una celda vecina si hay pasillo abierto: `'ARRIBA'`, `'ABAJO'`, `'IZQUIERDA'`, `'DERECHA'`. |
| Transición | Sumar el movimiento a la coordenada actual: `(r + dr, c + dc)`. |
| Prueba de objetivo | Comprobar si llegamos a la celda meta: `estado == (23, 20)`. |
| Costo de paso | En el laberinto normal cuesta $1.0$ cada paso. En la variante ponderada depende del terreno. |
| Costo de solución | La suma de los costos de cada paso desde el inicio hasta la meta. |

---

### Preguntas conceptuales

#### 1. Diferencia entre estado, nodo de búsqueda y celda dibujada
- Estado: Es solo el dato de la posición actual del agente (la tupla `(fila, columna)`). No sabe nada del pasado, ni de costos, ni de cómo llegó ahí.
- Nodo de búsqueda: Es el objeto que creamos en memoria durante el algoritmo. Contiene el estado actual, el apuntador al nodo `padre`, la acción con la que llegamos, el costo acumulado $g(n)$ y la profundidad.
- Celda dibujada: Es el recuadro visual o píxel que se pinta en la pantalla con Matplotlib. Es solo para visualización humana; al algoritmo no le afecta en nada.

#### 2. ¿Cuándo dos nodos distintos pueden tener el mismo estado?
Ocurre cuando el algoritmo llega a la misma celda por dos caminos distintos. Por ejemplo, en un laberinto con ciclos o en una búsqueda sin control de repetidos, podemos tener un nodo que llegó a `(5, 5)` por una ruta corta y otro nodo que llegó a la misma celda `(5, 5)` dando una vuelta más larga con diferente padre y mayor costo.

#### 3. ¿Por qué usamos búsqueda en grafo y no búsqueda en árbol?
Porque en una cuadrícula todos los pasillos son de ida y vuelta: si puedo ir de $A$ a $B$, también puedo devolverme de $B$ a $A$. Si hiciéramos búsqueda en árbol sin recordar qué celdas ya visitamos, el algoritmo se quedaría rebotando infinitamente entre dos celdas o generaría millones de caminos repetidos ($O(b^d)$). La búsqueda en grafo guarda los estados ya explorados y evita repetir trabajo.

#### 4. Invariantes que deben cumplirse durante la búsqueda
1. Frontera divisoria: La frontera siempre separa lo que ya exploramos de lo que aún no conocemos. Cualquier camino hacia una celda desconocida tiene que pasar por la frontera.
2. Trazabilidad: Si seguimos los punteros `padre` hacia atrás desde cualquier nodo, siempre llegamos de forma continua al inicio.
3. Monotonía en UCS: Como los costos son positivos, cada nodo que extraemos de la cola de prioridad siempre tiene un costo acumulado $g(n)$ igual o mayor al que sacamos antes.


## 4. Contrato común de las tres versiones

Todas las versiones y algoritmos (Desde cero, SimpleAI y AIMA-Python) deben satisfacer estrictamente el siguiente contrato de registro:

```python
@dataclass
class ResultadoBusqueda:
    encontrado: bool            # True si se alcanzó la meta, False en caso contrario
    camino: list[tuple[int,int]]# Secuencia completa de estados [inicio, ..., meta]
    acciones: list[str]         # Secuencia de acciones ['ABAJO', 'DERECHA', ...]
    costo: float                # Costo total acumulado del camino encontrado
    profundidad: int            # Profundidad del nodo meta en el árbol de búsqueda
    expandidos: int             # Nodos retirados de la frontera cuyos sucesores fueron generados
    generados: int              # Total de nodos sucesores construidos
    repetidos_descartados: int  # Sucesores descartados por ya pertenecer a visitados/alcanzados
    frontera_maxima: int        # Tamaño máximo simultáneo alcanzado por la frontera
    tiempo_ms: float            # Tiempo de ejecución en milisegundos
```

### Convenciones obligatorias aplicadas en el taller

- **Expandidos:** Se incrementa únicamente al retirar un nodo de la frontera y proceder a generar sus sucesores. Si el nodo meta es extraído y la búsqueda finaliza de inmediato, dicho nodo **no** se cuenta como expandido.
- **Generados:** Todo nodo sucesor construido se cuenta como generado, independientemente de que luego sea aceptado o descartado por duplicación. El nodo inicial se cuenta como el primer nodo generado ($1$).
- **Repetidos descartados:** Todo sucesor cuyo estado ya se encuentre en la estructura de alcanzados/cerrados (o en frontera con mejor costo) y no reemplace una entrada previa.
- **Frontera máxima:** Mide el pico de ocupación en memoria de la colección de frontera durante toda la ejecución.
- **Camino:** Lista ordenada conteniendo inicio y meta consecutivamente. Si no hay solución, `camino = []` y `encontrado = False`.


## 5. Pseudocódigo base y decisiones de diseño

### Respuestas a las preguntas de diseño

1. ¿Cuándo marcar un estado: al generarlo o al expandirlo?
   - En BFS (costos iguales a 1) conviene marcarlo al generarlo (apenas lo descubrimos y lo metemos a la cola). Si hacemos ahí mismo la prueba de meta, nos ahorramos guardar un nivel entero de nodos en la frontera y ganamos velocidad sin perder el camino óptimo.
   - En UCS, la meta obligatoriamente se comprueba al expandirlo (cuando sale de la cola de prioridad). Si lo hiciéramos al generarlo podríamos quedarnos con un camino que llegó primero a la meta pero que era carísimo.
   - En DFS, marcar al generar evita meter ciclos repetidos a la pila.

2. ¿Qué información mínima guarda cada nodo?
   Guardamos 5 cosas: `estado`, `padre`, `accion`, `costo_camino` y `profundidad`.

3. ¿Cómo reconstruimos el camino sin copiar listas en cada nodo?
   En vez de que cada nodo guarde una copia de toda la ruta recorrida (que gastaría un montón de memoria $O(d^2)$), cada nodo solo guarda un puntero a su `padre`. Cuando encontramos la meta, simplemente vamos saltando hacia atrás `actual = actual.padre` guardando los estados en una lista y al final la invertimos. Esto es súper rápido ($O(d)$).

4. ¿Cómo manejamos un camino más barato a un estado ya visitado?
   Guardamos un diccionario `mejor_costo[estado]`. Si descubrimos el mismo estado pero con un costo menor, actualizamos el diccionario y metemos el nuevo nodo a la cola de prioridad. Como actualizar un elemento dentro de un heap en Python es complicado, usamos la técnica de 'borrado perezoso' (lazy deletion): si al sacar un nodo del heap vemos que su costo es mayor al que tenemos registrado en `mejor_costo`, simplemente lo descartamos y seguimos.


# Versión 1 — Implementación desde cero

No puede utilizar bibliotecas de búsqueda o grafos. Se permiten únicamente estructuras estándar como listas, diccionarios, conjuntos, `deque` y colas de prioridad.

Debe implementar:

1. DFS iterativa;
2. BFS;
3. búsqueda de costo uniforme;
4. búsqueda limitada en profundidad;
5. profundización iterativa;
6. búsqueda bidireccional;
7. Lee o expansión por frente de onda.

La versión debe separar, como mínimo:

- representación del problema;
- representación de nodos;
- política de frontera;
- control de repetidos;
- reconstrucción del camino;
- recolección de métricas.

## 6. Pseudocódigo que debe traducirse

### DFS y BFS

```text
inicializar frontera con el nodo inicial
inicializar alcanzados

mientras haya nodos:
    extraer según disciplina LIFO o FIFO
    comprobar objetivo
    expandir
    insertar estados nuevos
```

La diferencia no debe quedar dispersa por todo el programa. Diseñe una abstracción que permita cambiar la política de frontera.

### Costo uniforme

```text
frontera ← cola de prioridad ordenada por g(n)
mejor_costo[inicial] ← 0

mientras frontera no esté vacía:
    extraer nodo con menor g
    ignorar entradas obsoletas
    comprobar objetivo
    para cada sucesor:
        nuevo_costo ← g(nodo) + costo_de_paso
        si mejora el costo conocido:
            actualizar costo, padre y frontera
```

### Profundización iterativa

```text
para límite = 0, 1, 2, ...:
    resultado ← BUSQUEDA-LIMITADA(problema, límite)
    si resultado es solución: devolverlo
    si resultado es fracaso definitivo: terminar
```

Debe distinguir **corte** de **fracaso**.

### Bidireccional

```text
crear una frontera desde inicio y otra desde meta
expandir de forma equilibrada
detectar intersección entre regiones alcanzadas
unir los dos caminos respetando su orientación
```

No se acepta ejecutar dos búsquedas completas y comparar al final.

In [ ]:
# VERSIÓN 1: BÚSQUEDA DESDE CERO (DFS, BFS, UCS, DLS, IDDFS, BIDIRECCIONAL)
from collections import deque
import heapq
import time
from typing import Callable, Dict, List, Optional, Set, Tuple, Union

# Importar estructuras modulares implementadas desde cero
from src.comun import ACCIONES_DELTA, DELTA_ACCIONES, Nodo, ResultadoBusqueda
from src.busqueda_cero import (
    ProblemaLaberinto, dfs_cero, bfs_cero, ucs_cero, 
    dls_wrapper, iddfs_cero, bidireccional_cero, lee_cero
)

# Instanciar el problema con el laberinto individual generado
problema_v1 = ProblemaLaberinto(grafo_individual, INICIO, META)

# Ejecutar todos los algoritmos requeridos de la Versión 1
resultados_v1 = {
    "DFS": dfs_cero(problema_v1),
    "BFS": bfs_cero(problema_v1),
    "UCS": ucs_cero(problema_v1),
    "DLS (L=150)": dls_wrapper(problema_v1, limite=150),
    "IDDFS": iddfs_cero(problema_v1, max_limite=200),
    "Bidireccional": bidireccional_cero(problema_v1),
}

# Presentar tabla comparativa estricta con todas las métricas del contrato
print("=== COMPARATIVA VERSIÓN 1: DESDE CERO (LABERINTO INDIVIDUAL) ===\n")
header = f"{'Algoritmo':<14} | {'Encontrado':<10} | {'Pasos':<6} | {'Costo':<6} | {'Expandidos':<10} | {'Generados':<10} | {'Descartados':<11} | {'FronteraMax':<11} | {'Tiempo (ms)':<10}"
print(header)
print("-" * len(header))
for nombre, res in resultados_v1.items():
    pasos = len(res.camino) - 1 if res.encontrado else 0
    print(f"{nombre:<14} | {str(res.encontrado):<10} | {pasos:<6} | {res.costo:<6.1f} | {res.expandidos:<10} | {res.generados:<10} | {res.repetidos_descartados:<11} | {res.frontera_maxima:<11} | {res.tiempo_ms:<10.3f}")


## 7. Lee: propagación de un frente luminoso

### Preguntas teóricas sobre el algoritmo de Lee

#### 1. ¿Por qué Lee es básicamente lo mismo que BFS en un laberinto sin pesos?
Porque Lee propaga una onda por turnos o 'pulsos'. En el turno 0 marca el inicio con etiqueta 0. En el turno 1 marca todos los vecinos a distancia 1. En el turno 2 marca los de distancia 2, y así sucesivamente. Esto es exactamente lo mismo que hace la cola FIFO de BFS por capas de profundidad. La etiqueta final que le queda a la meta en Lee es idéntica a la profundidad que encuentra BFS.

#### 2. ¿Qué pasa si los corredores tienen costos diferentes?
Lee asume que cada paso toma un 'tiempo' de 1 unidad. Si un corredor tiene lodo o costo 25 y otro tiene costo 1, la onda de Lee va a llegar primero por el camino que tenga menos celdas, etiquetándolo y cerrándolo antes de que llegue la onda por el camino que era más barato en costo acumulado. Para arreglar eso habría que usar una cola con prioridades por costo (que es básicamente convertir Lee en Dijkstra o UCS).


In [ ]:
# Implementación, Verificación y Visualización del Algoritmo de Lee
from src.visualizador_lee import visualizar_frente_lee, visualizar_mapa_calor_lee
from IPython.display import Image, display

# Ejecutar Lee registrando el historial completo de frentes de onda
res_lee, etiquetas_lee, frentes_lee = lee_cero(problema_v1, guardar_historial_frentes=True)

print("=== RESULTADO DEL ALGORITMO DE LEE ===")
print(f"Meta alcanzada: {res_lee.encontrado}")
print(f"Etiqueta asignada a la meta: {etiquetas_lee.get(META, 'No alcanzada')}")
print(f"Profundidad calculada por BFS: {resultados_v1['BFS'].profundidad}")
print(f"Longitud del camino (en aristas): {len(res_lee.camino) - 1}")

# Comprobación formal obligatoria
assert etiquetas_lee[META] == resultados_v1["BFS"].profundidad, \
    f"Error: discrepancia entre etiqueta de Lee ({etiquetas_lee[META]}) y BFS ({resultados_v1['BFS'].profundidad})"
print("\n✓ COMPROBACIÓN EXITOSA: La etiqueta de la meta en Lee coincide exactamente con la profundidad de BFS.")

# Generar y desplegar las gráficas obligatorias
visualizar_frente_lee(FILAS, COLUMNAS, grafo_individual, etiquetas_lee, frentes_lee, res_lee.camino, INICIO, META, guardar_como="lee_frente_onda.png")
visualizar_mapa_calor_lee(FILAS, COLUMNAS, etiquetas_lee, res_lee.camino, INICIO, META, guardar_como="lee_mapa_calor.png")

display(Image("lee_frente_onda.png"))
display(Image("lee_mapa_calor.png"))


# Versión 2 — SimpleAI

Para adaptar el laberinto a SimpleAI creamos la subclase `LaberintoSimpleAI(SearchProblem)`:
- `actions(state)`: devuelve los movimientos válidos según los pasillos del grafo.
- `result(state, action)`: calcula la nueva celda sumando el movimiento.
- `is_goal(state)`: revisa si llegamos a la meta.
- `cost(state, action, state2)`: devuelve el costo del paso.

### Sobre la instrumentación y diferencias de métricas
Para sacar las métricas que nos pide el taller sin tocar el código fuente instalado de SimpleAI, creamos una subclase de `BaseViewer` (`VisorMetricasSimpleAI`).

Diferencias que notamos frente a nuestra versión desde cero:
- Nodos visitados vs expandidos: SimpleAI cuenta como `visited_nodes` cada vez que saca un nodo de la frontera. Si ese nodo resulta ser la meta, para SimpleAI ya cuenta como visitado, mientras que en nuestro contrato un nodo meta no cuenta como expandido porque nunca le generamos hijos. Nuestro visor intercepta el evento `handle_expanded` para contar solo cuando de verdad se generan sucesores.
- Nodos generados: SimpleAI solo nos pasa al visor los hijos que pasaron su filtro interno de repetidos, mientras que en nuestra versión contamos todos los intentos de movimientos legales que calculamos.


In [ ]:
# VERSIÓN 2: MODELADO Y EJECUCIÓN CON SIMPLEAI
from simpleai.search import (
    SearchProblem,
    breadth_first,
    depth_first,
    uniform_cost,
    limited_depth_first,
    iterative_limited_depth_first,
)
from src.adaptador_simpleai import LaberintoSimpleAI, ejecutar_busqueda_simpleai

# Instanciar el SearchProblem de SimpleAI
problema_simpleai = LaberintoSimpleAI(grafo_individual, INICIO, META)

# Ejecutar los 5 algoritmos requeridos con graph_search=True e instrumentación externa
resultados_v2 = {
    "DFS (SimpleAI)": ejecutar_busqueda_simpleai(depth_first, problema_simpleai),
    "BFS (SimpleAI)": ejecutar_busqueda_simpleai(breadth_first, problema_simpleai),
    "UCS (SimpleAI)": ejecutar_busqueda_simpleai(uniform_cost, problema_simpleai),
    "DLS (SimpleAI, L=150)": ejecutar_busqueda_simpleai(limited_depth_first, problema_simpleai, depth_limit=150),
    "IDDFS (SimpleAI)": ejecutar_busqueda_simpleai(iterative_limited_depth_first, problema_simpleai),
}

# Presentar tabla comparativa
print("=== COMPARATIVA VERSIÓN 2: SIMPLEAI (LABERINTO INDIVIDUAL) ===\n")
header = f"{'Algoritmo':<22} | {'Encontrado':<10} | {'Pasos':<6} | {'Costo':<6} | {'Expandidos':<10} | {'Generados':<10} | {'FronteraMax':<11} | {'Tiempo (ms)':<10}"
print(header)
print("-" * len(header))
for nombre, res in resultados_v2.items():
    pasos = len(res.camino) - 1 if res.encontrado else 0
    print(f"{nombre:<22} | {str(res.encontrado):<10} | {pasos:<6} | {res.costo:<6.1f} | {res.expandidos:<10} | {res.generados:<10} | {res.frontera_maxima:<11} | {res.tiempo_ms:<10.3f}")


# Versión 3 — AIMA-Python

Para la versión con AIMA-Python creamos la subclase `LaberintoAIMA(Problem)` implementando `actions`, `result`, `goal_test` y `path_cost`.

### Exigencia avanzada y análisis de la biblioteca

#### 1. ¿Por qué contar llamadas a `actions` no es lo mismo que contar nodos generados?
Porque cuando AIMA va a expandir un nodo, llama a `actions(estado)` una sola vez. Esa llamada devuelve una lista con las acciones posibles (por ejemplo, 3 movimientos). Luego, por cada una de esas 3 acciones, llama a `result` y crea un nodo hijo. Es decir, una sola llamada a `actions` produce varios nodos generados. Contar llamadas a `actions` nos dice cuántas veces expandimos, no cuántos hijos nacieron.

#### 2. Comparación entre el `Node` de AIMA y nuestro `Nodo` propio
- Ambos guardan lo básico: estado, padre, acción, costo y profundidad.
- El nodo de AIMA tiene métodos útiles como `expand()`, `solution()` y `path()`.
- Nuestro nodo propio lo hicimos más ligero usando `slots` para que consuma menos memoria en laberintos grandes, y le pusimos el operador `<` (`lt`) para que entre directo en un `heapq` de Python sin envoltorios extra.


In [ ]:
# VERSIÓN 3: MODELADO Y EJECUCIÓN CON AIMA-PYTHON
import aima.search as asrch
from src.adaptador_aima import LaberintoAIMA, ejecutar_busqueda_aima

# Ejecutar los algoritmos comparables de AIMA-Python
resultados_v3 = {
    "BFS (AIMA)": ejecutar_busqueda_aima(asrch.breadth_first_graph_search, grafo_individual, INICIO, META),
    "DFS (AIMA)": ejecutar_busqueda_aima(asrch.depth_first_graph_search, grafo_individual, INICIO, META),
    "UCS (AIMA)": ejecutar_busqueda_aima(asrch.uniform_cost_search, grafo_individual, INICIO, META),
    "DLS (AIMA, L=150)": ejecutar_busqueda_aima(asrch.depth_limited_search, grafo_individual, INICIO, META, limit=150),
    "Bidireccional (AIMA)": ejecutar_busqueda_aima(asrch.bidirectional_search, grafo_individual, INICIO, META, evitar_retroceso=False),
}

# Presentar tabla comparativa
print("=== COMPARATIVA VERSIÓN 3: AIMA-PYTHON (LABERINTO INDIVIDUAL) ===\n")
header = f"{'Algoritmo':<22} | {'Encontrado':<10} | {'Pasos':<6} | {'Costo':<6} | {'Expandidos':<10} | {'Generados':<10} | {'Tiempo (ms)':<10}"
print(header)
print("-" * len(header))
for nombre, res in resultados_v3.items():
    pasos = len(res.camino) - 1 if res.encontrado and res.camino else res.profundidad
    print(f"{nombre:<22} | {str(res.encontrado):<10} | {pasos:<6} | {res.costo:<6.1f} | {res.expandidos:<10} | {res.generados:<10} | {res.tiempo_ms:<10.3f}")


## 8. Pruebas de aceptación y verificación formal

Batería de pruebas automatizadas para garantizar la corrección matemática y algorítmica de todas las soluciones:

### 1. Criterios de validez de un camino
Todo resultado exitoso debe certificar:
1. `camino[0] == inicio` (condición de frontera inicial).
2. `camino[-1] == meta` (condición de frontera final).
3. Para todo $i$: $(camino[i], camino[i+1]) \in E$ (continuidad estricta en corredores existentes).
4. El costo acumulado reportado coincide exactamente con $\sum c(s_i, a_i, s_{i+1})$.
5. Re-ejecutar secuencialmente las acciones `acciones` reproduce idénticamente la lista de estados `camino`.

### 2. Reglas de concordancia entre versiones
- **Costos unitarios:** $\text{longitud}(\text{BFS}) = \text{costo}(\text{UCS}) = \text{etiqueta}_{\text{meta}}(\text{Lee})$.
- **En topología de árbol:** Dado que existe un único camino simple entre dos nodos en cualquier árbol, todas las soluciones válidas generadas por BFS, DFS, UCS, DLS, IDDFS, Bidireccional y Lee deben coincidir en la secuencia exacta de estados.
- **Concordancia entre las 3 versiones:** Las tres bibliotecas (Desde cero, SimpleAI y AIMA-Python) deben reportar la misma existencia de solución y el mismo costo óptimo.

### 3. Batería de casos límite obligatorios
- Inicio igual a meta ($s_0 == s_\text{meta}$).
- Laberinto mínimo de $1 \times 1$.
- Meta inalcanzable (grafo particionado desconexo).
- Múltiples entradas obsoletas en UCS con actualización por caminos de menor costo.
- DLS evaluado con límite insuficiente ($L < d$), límite exacto ($L = d$) y límite holgado ($L > d$).


In [ ]:
# EJECUCIÓN DE LA BATERÍA COMPLETA DE PRUEBAS DE ACEPTACIÓN
from src.pruebas_aceptacion import BateriaPruebasAceptacion, ValidadorCamino
from src.busqueda_cero import ProblemaLaberinto, bfs_cero, dfs_cero, ucs_cero, lee_cero

print("=== INICIANDO BATERÍA DE PRUEBAS DE ACEPTACIÓN Y CASOS LÍMITE ===\n")

# 1. Validación de caminos de las soluciones obtenidas en el laberinto individual
prob_ind = ProblemaLaberinto(grafo_individual, INICIO, META)
for nombre_algo, res in resultados_v1.items():
    valido, mensaje = ValidadorCamino.validar(prob_ind, res)
    print(f"[Validador de Camino] {nombre_algo:<14}: {'✓ OK' if valido else '✗ ERROR'} ({mensaje})")
    assert valido, f"Fallo en validador de camino para {nombre_algo}"

print("\n--- 2. Pruebas de Concordancia en Árbol ---")
ok_arbol, msg_arbol = BateriaPruebasAceptacion.probar_concordancia_arbol(grafo_individual, INICIO, META)
print(f"[Concordancia en árbol]: {'✓ OK' if ok_arbol else '✗ ERROR'} - {msg_arbol}")

print("\n--- 3. Casos Límite Obligatorios ---")
ok_c1, msg_c1 = BateriaPruebasAceptacion.probar_caso_inicio_igual_meta()
print(f"[Caso Límite 1] Inicio == Meta: {'✓ OK' if ok_c1 else '✗ ERROR'} - {msg_c1}")

ok_c2, msg_c2 = BateriaPruebasAceptacion.probar_caso_laberinto_1x1()
print(f"[Caso Límite 2] Laberinto 1x1:  {'✓ OK' if ok_c2 else '✗ ERROR'} - {msg_c2}")

ok_c3, msg_c3 = BateriaPruebasAceptacion.probar_caso_meta_inalcanzable()
print(f"[Caso Límite 3] Meta Inalcanzable: {'✓ OK' if ok_c3 else '✗ ERROR'} - {msg_c3}")

ok_c4, msg_c4 = BateriaPruebasAceptacion.probar_caso_entradas_obsoletas_ucs()
print(f"[Caso Límite 4] Entradas obsoletas en UCS: {'✓ OK' if ok_c4 else '✗ ERROR'} - {msg_c4}")

prof_real = len(resultados_v1["BFS"].camino) - 1
ok_c5, msg_c5 = BateriaPruebasAceptacion.probar_caso_limites_dls(grafo_individual, INICIO, META, prof_real)
print(f"[Caso Límite 5] Límites DLS (insuficiente, exacto, excesivo): {'✓ OK' if ok_c5 else '✗ ERROR'} - {msg_c5}")

print("\n>>> TODAS LAS PRUEBAS DE ACEPTACIÓN PASARON SATISFACTORIAMENTE <<<")


## 9. Segunda familia de problemas: ciclos y costos ponderados

Como el algoritmo Hunt-and-Kill genera un árbol, siempre hay un solo camino posible y todos los algoritmos terminan sacando la misma ruta. Para ver de verdad cómo se diferencian, hicimos dos pruebas:

### A. Abrir paredes para meter ciclos (8% de paredes)
Buscamos paredes internas entre celdas vecinas que estaban bloqueadas y abrimos el 8% de ellas al azar con semilla controlada.
- Al meter ciclos, ahora hay múltiples caminos para llegar a la meta.
- Aquí se nota clarísimo la diferencia: BFS encuentra el camino más corto (62 pasos), mientras que DFS se pierde por ramas largas y saca un camino de 132 pasos.

### B. Laberinto con costos (Dilema pasos vs costo)
Pusimos un camino corto de 2 pasos pero muy pesado (costo 25 cada paso = 50 en total), y otro camino más largo de 4 pasos pero despejado (costo 1 cada paso = 4 en total).
- BFS solo cuenta pasos, así que se va por el atajo pesado de 2 pasos (costo 50.0).
- UCS mira los costos acumulados, así que prefiere dar la vuelta de 4 pasos para gastar solo 4.0.
Con esto comprobamos que *el camino con menos pasos no siempre es el más barato*.


In [ ]:
# IMPLEMENTACIÓN DE CICLOS Y COSTOS PONDERADOS
from src.topologias_costos import agregar_ciclos, detectar_ciclos_dfs, crear_escenario_dilema_costos
from src.busqueda_cero import ProblemaLaberinto, bfs_cero, dfs_cero, ucs_cero

print("=== TRANSFORMACIÓN A: LABERINTO CON CICLOS (8% PAREDES ABIERTAS) ===\n")
# Aplicar apertura del 8% de paredes sobre el laberinto individual
grafo_ciclos, num_abiertas, paredes_abiertas = agregar_ciclos(
    grafo_individual, FILAS, COLUMNAS, porcentaje=0.08, semilla=SEMILLA
)
ciclos_detectados = detectar_ciclos_dfs(grafo_ciclos)

print(f"Total paredes internas abiertas: {num_abiertas}")
print(f"Aristas totales: {sum(len(v) for v in grafo_ciclos.values()) // 2} (Árbol base: {FILAS*COLUMNAS - 1})")
print(f"Ciclos verificados algorítmicamente (muestra): {len(ciclos_detectados)} ciclos")

# Comparar BFS vs DFS en presencia de ciclos
prob_ciclos = ProblemaLaberinto(grafo_ciclos, INICIO, META)
res_bfs_ciclos = bfs_cero(prob_ciclos)
res_dfs_ciclos = dfs_cero(prob_ciclos)

print(f"\n[Topología con ciclos] BFS: Pasos = {len(res_bfs_ciclos.camino)-1}, Tiempo = {res_bfs_ciclos.tiempo_ms:.2f} ms")
print(f"[Topología con ciclos] DFS: Pasos = {len(res_dfs_ciclos.camino)-1}, Tiempo = {res_dfs_ciclos.tiempo_ms:.2f} ms")
assert len(res_bfs_ciclos.camino) <= len(res_dfs_ciclos.camino), "BFS debe ser óptimo en pasos."
print("✓ Comprobado: BFS encuentra un camino estrictamente más corto en pasos que DFS ante la presencia de ciclos.")

print("\n=== TRANSFORMACIÓN B: LABERINTO PONDERADO (DILEMA PASOS VS COSTO) ===\n")
grafo_pond, ini_p, meta_p, costo_fn = crear_escenario_dilema_costos()
prob_pond = ProblemaLaberinto(grafo_pond, ini_p, meta_p, costo_arista_fn=costo_fn)

res_bfs_pond = bfs_cero(prob_pond)
res_ucs_pond = ucs_cero(prob_pond)

print(f"Solución BFS (minimiza pasos):  Pasos = {len(res_bfs_pond.camino)-1} | Costo = {res_bfs_pond.costo}")
print(f"Solución UCS (minimiza costos): Pasos = {len(res_ucs_pond.camino)-1} | Costo = {res_ucs_pond.costo}")

assert len(res_bfs_pond.camino) < len(res_ucs_pond.camino), "BFS debió encontrar menos pasos"
assert res_ucs_pond.costo < res_bfs_pond.costo, "UCS debió encontrar menor costo acumulado"
print("\n✓ DILEMA VALIDADO EXITOSAMENTE: camino con menos pasos ≠ camino de menor costo.")


## 10. Discusión de los resultados experimentales

#### 1. ¿Qué métrica es más confiable que el tiempo en milisegundos?
Definitivamente los nodos expandidos y los nodos generados. El tiempo en milisegundos cambia cada vez que uno corre el código dependiendo de si el computador tiene otros programas abiertos, de la recolección de basura de Python o de la memoria caché. En cambio, los nodos expandidos dependen puramente de la lógica del algoritmo y dan exactamente el mismo número cada vez.

#### 2. ¿Cuándo la búsqueda bidireccional ya no vale la pena?
Pierde su ventaja cuando el laberinto tiene pasillos muy largos y angostos con pocas bifurcaciones ($b \approx 1$). Ahí la sobrecarga de alternar entre los dos frentes y revisar a cada rato si ya chocaron termina tomando más tiempo que simplemente lanzar un BFS normal. También se complica si hay muchas metas posibles o si los dos frentes se desvían por zonas separadas.

#### 3. ¿Por qué IDDFS repite trabajo y aun así es una buena opción?
Porque en un árbol de búsqueda, casi todos los nodos están en el último nivel. Los niveles anteriores son solo una pequeña fracción de los nodos totales. Repetirlos cuesta muy poco tiempo extra, pero a cambio nos da una ventaja gigante: encuentra la solución óptima como BFS pero gastando poquísima memoria RAM como DFS (memoria lineal $O(b \cdot d)$ en vez de exponencial $O(b^d)$).

#### 4. ¿Cómo afecta el orden de los vecinos a DFS?
En DFS el orden lo es todo. Si la primera acción que prueba (por ejemplo, ir hacia abajo o hacia la derecha) apunta por casualidad hacia donde está la meta, DFS llega volando en línea recta. Pero si primero prueba una dirección que lleva a un callejón largo en la dirección contraria, se va a meter hasta el fondo y va a tardar muchísimo en corregir el rumbo.

#### 5. ¿Qué diferencias vienen del algoritmo y cuáles de la biblioteca?
- La cantidad de nodos expandidos y la longitud del camino dependen del algoritmo (si es FIFO, LIFO o cola de prioridad).
- El tiempo en milisegundos depende de cómo está programada la biblioteca. Nuestra versión desde cero corre más rápido porque no tiene tantas capas de clases ni verificaciones internas. SimpleAI tiene más clases y observadores que consumen tiempo de CPU, y AIMA crea muchos objetos `Node` intermedios.


In [ ]:
# PROTOCOLO EXPERIMENTAL, CÁLCULO DE MEDIANAS / IQR Y DESPLIEGUE DE GRÁFICAS
from src.protocolo_experimental import (
    cargar_o_ejecutar_protocolo,
    calcular_tabla_iqr,
    generar_las_seis_graficas,
)
from IPython.display import Image, display

# 1. Cargar o ejecutar el protocolo experimental factorial (carga instantánea)
df_experimentos = cargar_o_ejecutar_protocolo(LaberintoHuntKill, n_repeticiones=10)

# 2. Resumen estadístico: Mediana e IQR por tamaño y algoritmo
tabla_iqr = calcular_tabla_iqr(df_experimentos)
print("=== TABLA RESUMEN ESTADÍSTICO (MEDIANA E IQR) ===\n")
print(tabla_iqr.head(15).to_string(index=False))

# 3. Asegurar las 6 gráficas obligatorias
generar_las_seis_graficas(df_experimentos, LaberintoHuntKill, FILAS, COLUMNAS, SEMILLA)

# 4. Desplegar visualmente las 6 figuras generadas
print("\n--- 1. Nodos Expandidos vs Número de Estados (|V|) ---")
display(Image("grafica1_expandidos_vs_estados.png"))

print("--- 2. Frontera Máxima vs Profundidad de la Solución ---")
display(Image("grafica2_frontera_vs_profundidad.png"))

print("--- 3. Tiempo de Búsqueda frente a Tamaño ---")
display(Image("grafica3_tiempo_vs_tamano.png"))

print("--- 4. Costo y Longitud de Soluciones por Topología ---")
display(Image("grafica4_costo_y_longitud.png"))

print("--- 5. Mapa de Calor de Lee ---")
display(Image("grafica5_mapa_calor_lee.png"))

print("--- 6. Comparación Cruzada entre las Tres Versiones ---")
display(Image("grafica6_comparativa_versiones.png"))


## 11. Análisis teórico y conexión empírica

Tabla comparativa de complejidad asintótica y propiedades fundamentales según Russell & Norvig (AIMA, Cap. 3), donde:
- $b$: factor de ramificación efectivo ($b \le 4$ en cuadrículas ortogonales).
- $d$: profundidad del nodo objetivo óptimo.
- $m$: profundidad máxima del espacio de estados.
- $C^*$: costo de la solución óptima; $\epsilon > 0$: cota inferior de costo de arista.

| Algoritmo | Completo | Óptimo | Tiempo | Espacio | Condiciones de Validez |
|---|:---:|:---:|:---:|:---:|---|
| DFS | Sí (en grafos finitos con visitados) | No | $O(|V| + |E|) \approx O(b^m)$ | $O(b \cdot m)$ (árbol) / $O(|V|)$ (grafo) | Requiere control de ciclos o grafo acíclico finito. |
| BFS | Sí (si $b < \infty$) | Sí (en costos unitarios) | $O(b^d) \approx O(|V| + |E|)$ | $O(b^d)$ (almacena la frontera en memoria) | Costos de paso idénticos/unitarios para optimalidad. |
| UCS | Sí (si $c \ge \epsilon > 0$) | Sí | $O(b^{1 + \lfloor C^*/\epsilon \rfloor})$ | $O(b^{1 + \lfloor C^*/\epsilon \rfloor})$ | Costos no negativos acotados por $\epsilon > 0$. |
| DLS | Sí (si $d \le L$) | No | $O(b^L)$ | $O(b \cdot L)$ | Requiere conocer de antemano una cota $L \ge d$. |
| IDDFS | Sí (si $b < \infty$) | Sí (en costos unitarios) | $O(b^d)$ | $O(b \cdot d)$ | Combina optimalidad de BFS con memoria lineal de DFS. |
| Bidireccional | Sí (si $b < \infty$) | Sí (en costos unitarios) | $O(b^{d/2})$ | $O(b^{d/2})$ | Meta única, conocida y operadores inversibles. |
| Lee | Sí (en grafos finitos) | Sí (en costos unitarios) | $O(|V| + |E|)$ | $O(|V|)$ | Costos unitarios homogéneos en cuadrícula. |

---

### Conexión entre las cotas teóricas y las mediciones experimentales

1. Crecimiento de Nodos Expandidos (Gráfica 1):
   - En BFS, UCS y Lee, la cantidad de nodos expandidos escala linealmente con respecto al orden $|V|$ del laberinto ($O(|V|)$ en búsqueda en grafo), confirmando que ningún estado alcanzado se expande más de una vez.
   - La búsqueda Bidireccional reduce los nodos expandidos significativamente frente a BFS unidireccional cuando la meta se ubica en el interior o cuadrantes opuestos, validando la reducción exponencial de radio de búsqueda $\approx b^{d/2} + b^{d/2} \ll b^d$.

2. Comportamiento de la Frontera Máxima (Gráfica 2):
   - En DFS, la frontera máxima se mantiene extraordinariamente pequeña ($O(b \cdot m) \le 15$ nodos en grillas de hasta 1600 celdas), corroborando su bajo consumo de memoria.
   - En BFS, la frontera crece proporcionalmente al ancho de la capa frontal (frente de onda), alcanzando picos proporcionales al perímetro del árbol.

3. Optimalidad y Discrepancias Topológicas (Gráfica 4):
   - En el laberinto tipo árbol (Hunt-and-Kill), la longitud de las rutas halladas por BFS, DFS, UCS, DLS y Lee es idéntica ($128$ pasos en la instancia individual), debido al teorema de unicidad de caminos simples en árboles.
   - Al introducir ciclos ($5\%$ y $10\%$), BFS y UCS encuentran caminos de menor longitud ($62$ pasos frente a $132$ pasos en DFS), confirmando que DFS carece de optimalidad.
   - En el laberinto ponderado, se evidenció formalmente que $\text{costo}(\text{UCS}) < \text{costo}(\text{BFS})$, certificando que solo UCS minimiza el costo acumulado cuando los pasos no son homogéneos.


## 12. Respuestas a las preguntas para la defensa oral

#### 1. Muestre en memoria la diferencia entre frontera y alcanzados
#### 2. Cambie BFS a DFS modificando solamente la política apropiada
#### 3. Explique un caso en el que marcar visitados demasiado tarde duplique trabajo

#### 4. Explique por qué terminar UCS al generar la meta puede ser incorrecto

#### 5. Reconstruya un camino usando padres sin almacenar caminos completos
#### 6. Muestre por qué dos BFS bidireccionales requieren orientar correctamente los padres
#### 7. Explique por qué Lee no es un rayo geométrico

#### 8. Adapte el problema a una meta múltiple

#### 9. Introduzca una celda bloqueada durante la ejecución y analice qué debe recalcularse

#### 10. Compare `SearchProblem` de SimpleAI con `Problem` de AIMA-Python

#### 11. Diagnostique una discrepancia de métricas entre dos versiones


#### 12. Argumente por qué un laberinto perfecto oculta diferencias de calidad de ruta


## 13. Entrega y verificación de la rúbrica

### Matriz de cumplimiento de criterios evaluativos

| Criterio | Peso | Estado de Cumplimiento |
|---|:---:|---|
| Modelado, invariantes y auditoría de Hunt-and-Kill | 10% | ✓ Demostración formal de conectividad, $|V|-1$ aristas, propiedad de árbol y batería automática de 6 pruebas de auditoría. |
| Versión desde cero y estructuras de datos | 25% | ✓ Implementación modular desacoplada de DFS, BFS, UCS, DLS, IDDFS, Bidireccional y Lee con dataclass `ResultadoBusqueda`. |
| Adaptación correcta a SimpleAI | 12% | ✓ Subclase `SearchProblem` con instrumentación externa `BaseViewer` sin alterar la biblioteca. |
| Adaptación correcta a AIMA-Python | 13% | ✓ Subclase `Problem` con soporte para algoritmos de grafo y árbol, comparativa de nodos e instrumentación pura. |
| Pruebas, casos límite y concordancia | 15% | ✓ Batería completa: validador de caminos, concordancia de versiones, $1\times 1$, meta inalcanzable, entradas obsoletas y límites de DLS. |
| Experimentos, métricas y visualización | 12% | ✓ Protocolo factorial con semillas controladas, cálculo de medianas e IQR, y generación de las 6 gráficas obligatorias. |
| Análisis teórico y conclusiones | 8% | ✓ Tabla asintótica completa ($b, d, m$), conexión empírica y respuestas exhaustivas a las 12 preguntas de defensa. |
| Defensa, trazabilidad y calidad del cuaderno | 5% | ✓ Historial ordenado de commits descriptivos en Git, reproducibilidad de punta a punta y cuaderno limpio. |

Enlace al repositorio de trabajo en GitHub: <https://github.com/Valdefer54/Hunt-and-Kill-y-Busqueda-No-Informada>


## 14. Referencias bibliográficas

1. **Russell, S. J., & Norvig, P.** (2020). *Artificial Intelligence: A Modern Approach* (4th ed.). Pearson. Capítulo 3: "Solving Problems by Searching".
2. **Lee, C. Y.** (1961). "An Algorithm for Path Connections and Its Applications". *IRE Transactions on Electronic Computers*, EC-10(3), 346–365. <https://doi.org/10.1109/TEC.1961.5219222>
3. **Adamatzky, A.** (2016). "Physical maze solvers. All twelve prototypes implement 1961 Lee algorithm". *arXiv preprint*. <https://arxiv.org/abs/1601.04672>
4. **SimpleAI Team.** (2022). *Simple AI: Python library that implements many Artificial Intelligence algorithms*. Repositorio GitHub: <https://github.com/simpleai-team/simpleai>
5. **Norvig, P. et al.** (2022). *AIMA Python: Python implementation of algorithms from Russell And Norvig's Artificial Intelligence: A Modern Approach*. Repositorio GitHub: <https://github.com/aimacode/aima-python>
6. **Cormen, T. H., Leiserson, C. E., Rivest, R. L., & Stein, C.** (2009). *Introduction to Algorithms* (3rd ed.). MIT Press. Capítulos 22 ("Elementary Graph Algorithms") y 23 ("Minimum Spanning Trees").
